# Explainable Quantum Machine Learning Dashboard

This executable walkthrough follows one saved prediction from benchmark evidence to encoded angles, circuit dynamics, local gradients, and a bounded counterfactual.

**Created by School of AI and School of QC.**

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from explainable_qml_dashboard.data import FEATURE_NAMES, TARGET_NAMES
from explainable_qml_dashboard.explain import (
    circuit_trace_frames,
    encoding_frame,
    local_input_saliency,
)
from explainable_qml_dashboard.inference import load_inference_bundle
from explainable_qml_dashboard.quantum import build_qiskit_circuit

ROOT = Path.cwd()
if not (ROOT / "examples" / "reference-run").exists():
    ROOT = ROOT.parent
RUN = ROOT / "examples" / "reference-run"
assert RUN.exists(), "Run this notebook from the repository or notebooks directory."

## 1. Inspect the frozen-test benchmark

The classical controls and QVC use the same train-fitted angle representation and frozen test rows.

In [ ]:
metrics = pd.read_csv(RUN / "summary_metrics.csv")
metrics[["model", "accuracy", "macro_f1", "log_loss", "expected_calibration_error"]]

In [ ]:
ax = metrics.sort_values("macro_f1").plot.barh(
    x="model", y="macro_f1", legend=False, color="#5B4BDB", figsize=(8, 4)
)
ax.set(xlabel="Frozen-test macro F1", ylabel="", xlim=(0, 1), title="Matched model comparison")
plt.show()

## 2. Encode one frozen test row

The scaler was fitted only on training rows. Each of the four resulting angles controls one qubit.

In [ ]:
bundle = load_inference_bundle(RUN)
dataset = pd.read_csv(RUN / "dataset_snapshot.csv")
split = pd.read_csv(RUN / "fixed_split.csv")
row_id = int(split.loc[split["split"] == "test", "row_id"].iloc[0])
raw = dataset.loc[dataset["row_id"] == row_id, list(FEATURE_NAMES)].to_numpy(dtype=float)[0]
angles = bundle.preprocessor.transform(raw[None, :])[0]
encoding = encoding_frame(raw, angles, FEATURE_NAMES)
encoding[["feature", "raw_value", "angle_radians", "angle_fraction_of_pi"]]

## 3. Trace circuit behavior

The same saved parameters feed the transparent simulator and an independently built Qiskit circuit.

In [ ]:
trace, basis = circuit_trace_frames(angles, bundle.quantum_model)
trace

In [ ]:
probability_columns = [f"probability_{name}" for name in TARGET_NAMES]
ax = trace.set_index("stage")[probability_columns].plot(
    marker="o", figsize=(9, 4), color=["#5B4BDB", "#14A38B", "#F59E0B"]
)
ax.set(ylabel="Class probability", xlabel="", ylim=(0, 1), title="Prediction through the circuit")
plt.xticks(rotation=15)
plt.show()

In [ ]:
circuit = build_qiskit_circuit(angles, bundle.quantum_model.parameters)
print(circuit.draw(output="text", fold=110))

## 4. Explain the prediction

Input saliency is the signed local derivative of the predicted-class probability with respect to an encoded angle. It is not causal importance.

In [ ]:
probabilities = bundle.quantum_model.predict_proba(angles[None, :])[0]
prediction = int(probabilities.argmax())
print(f"Prediction: {TARGET_NAMES[prediction]}")
print(dict(zip(TARGET_NAMES, probabilities.round(4), strict=True)))
local_input_saliency(bundle.quantum_model, angles, FEATURE_NAMES)

In [ ]:
counterfactual = pd.read_json(RUN / "counterfactual.json", typ="series")
counterfactual

## 5. Audit the training trace

Every trainable angle has an exact parameter-shift derivative for every completed epoch.

In [ ]:
history = pd.read_csv(RUN / "training_history.csv")
gradients = pd.read_csv(RUN / "parameter_gradients.csv")
print(f"Recorded {len(gradients):,} parameter gradients across {len(history)} epochs.")
history.tail()

## Conclusion

The classical baselines win this frozen comparison. The quantum model remains useful here because its complete mechanism—from angles to state dynamics to gradients and predictions—is available for inspection. That is the project's claim boundary.

**Created by School of AI and School of QC.**